# [빠른 버전] ResNet-34 / ResNet-50 Ablation Study (CIFAR-10, Colab GPU 기준 수 분 내 완료)

이전 노트북(224x224, ImageNet 스타일 stem)은 "정석 구조"로 그대로 두고,
이 노트북은 **속도 최적화**에 집중한 버전입니다. 변경한 부분:

1. **CIFAR 전용 stem** (`cifar_stem=True`): `7x7 stride2 + maxpool` → `3x3 stride1`, maxpool 제거
   → 32x32 입력을 그대로 사용 (224로 리사이즈 안 함 → 연산량 약 49배 감소)
2. **subset 샘플링**: 전체 5만장 대신 일부만 사용해도 plain vs residual 경향성은 충분히 드러남
3. **Mixed Precision(AMP)**: GPU에서 자동으로 활성화되어 추가로 1.5~2배 가속

> Colab에서 **런타임 > 런타임 유형 변경 > GPU**로 설정하고 실행하세요.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type != 'cuda':
    print('⚠️  GPU가 잡히지 않았습니다. Colab 상단 메뉴에서 런타임 유형을 GPU로 변경하세요.')


## 1. 모델 구현 (이전과 동일 + `cifar_stem` 옵션 추가)

`cifar_stem=True`로 설정하면 32x32 입력에 맞게 stem을 가볍게 바꿉니다. 블록 구조(BasicBlock/Bottleneck, plain/residual 토글)는 이전 노트북과 완전히 동일합니다.


In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None, use_residual=True):
        super().__init__()
        self.use_residual = use_residual
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.use_residual:
            if self.downsample is not None:
                identity = self.downsample(x)
            out = out + identity
        return self.relu(out)


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1, downsample=None, use_residual=True):
        super().__init__()
        self.use_residual = use_residual
        self.conv1 = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.use_residual:
            if self.downsample is not None:
                identity = self.downsample(x)
            out = out + identity
        return self.relu(out)


In [ ]:
class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=10, use_residual=True, cifar_stem=False):
        super().__init__()
        self.in_channels = 64
        self.use_residual = use_residual

        if cifar_stem:
            # CIFAR(32x32)용 가벼운 stem: 다운샘플 없이 채널만 64로
            self.stem = nn.Sequential(
                nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False),
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True),
            )
        else:
            # ImageNet(224x224) 스타일 stem
            self.stem = nn.Sequential(
                nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            )

        self.layer1 = self._make_layer(block, 64, layers[0], stride=1)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        downsample = None
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * block.expansion, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * block.expansion),
            )
        layers = [block(self.in_channels, out_channels, stride, downsample, use_residual=self.use_residual)]
        self.in_channels = out_channels * block.expansion
        for _ in range(1, num_blocks):
            layers.append(block(self.in_channels, out_channels, use_residual=self.use_residual))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


def resnet34(num_classes=10, residual=True, cifar_stem=True):
    return ResNet(BasicBlock, [3, 4, 6, 3], num_classes=num_classes,
                  use_residual=residual, cifar_stem=cifar_stem)

def resnet50(num_classes=10, residual=True, cifar_stem=True):
    return ResNet(Bottleneck, [3, 4, 6, 3], num_classes=num_classes,
                  use_residual=residual, cifar_stem=cifar_stem)


## 2. 데이터 준비 — 32x32 그대로 사용 + subset 샘플링

`TRAIN_SUBSET` / `VAL_SUBSET`을 조절해서 속도와 신뢰도를 트레이드오프하세요.
빠른 확인용: 각 5,000 / 1,000 정도면 충분히 경향성이 보입니다. 제출용 최종본은 값을 늘리세요(예: 전체 사용).


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616]),
])

full_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
full_val = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

TRAIN_SUBSET = 5000   # None으로 두면 전체(50000) 사용
VAL_SUBSET = 1000     # None으로 두면 전체(10000) 사용
SEED = 42

rng = np.random.default_rng(SEED)

def make_subset(dataset, n):
    if n is None or n >= len(dataset):
        return dataset
    idx = rng.choice(len(dataset), size=n, replace=False)
    return Subset(dataset, idx.tolist())

train_set = make_subset(full_train, TRAIN_SUBSET)
val_set = make_subset(full_val, VAL_SUBSET)

BATCH_SIZE = 128
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print('train:', len(train_set), 'val:', len(val_set))


## 3. 학습 함수 (Mixed Precision 적용)

In [ ]:
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
    return running_loss / len(loader.dataset), correct / len(loader.dataset)


def train_model(model_fn, residual, epochs, lr=1e-3):
    model = model_fn(num_classes=10, residual=residual, cifar_stem=True).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        print(f'  epoch {epoch+1}/{epochs} train_loss={train_loss:.4f} '
              f'val_loss={val_loss:.4f} val_acc={val_acc:.4f} ({time.time()-t0:.1f}s)')
    return history


## 4. Ablation Study 실행

`EPOCHS`를 늘릴수록 결과가 더 신뢰할 만해집니다. 우선 빠르게 3~5로 시작해서 파이프라인을 확인한 뒤 늘려보세요.


In [ ]:
EPOCHS = 5

configs = [
    ('ResNet-34', 'Plain',    resnet34, False),
    ('ResNet-34', 'Residual', resnet34, True),
    ('ResNet-50', 'Plain',    resnet50, False),
    ('ResNet-50', 'Residual', resnet50, True),
]

results = {}
for arch_name, variant_name, model_fn, use_residual in configs:
    print(f'\n===== {arch_name} - {variant_name} =====')
    results[f'{arch_name}_{variant_name}'] = train_model(model_fn, use_residual, epochs=EPOCHS)


In [ ]:
rows = []
for arch_name, variant_name, model_fn, use_residual in configs:
    key = f'{arch_name}_{variant_name}'
    rows.append({
        'Model': arch_name,
        'Depth': 34 if arch_name == 'ResNet-34' else 50,
        'Residual 여부': variant_name,
        'Final Val Accuracy': round(results[key]['val_acc'][-1], 4),
        'Epochs': EPOCHS,
        'Train Subset': len(train_set),
        'Val Subset': len(val_set),
    })

ablation_df = pd.DataFrame(rows)
ablation_df.to_csv('ablation_study_results.csv', index=False)
print('저장 완료: ablation_study_results.csv')
ablation_df


In [ ]:
plt.figure(figsize=(6, 4))
for arch_name, variant_name, model_fn, use_residual in configs:
    key = f'{arch_name}_{variant_name}'
    style = '-' if variant_name == 'Residual' else '--'
    plt.plot(range(1, EPOCHS + 1), results[key]['val_acc'], style, marker='o',
              label=f'{arch_name} ({variant_name})')
plt.xlabel('epoch'); plt.ylabel('validation accuracy')
plt.title('Ablation Study (fast, subset): Plain vs Residual')
plt.legend(); plt.grid(alpha=0.3); plt.show()
